# Export Classification Reason to Phy

This notebook extracts the main classification reason for each unit from Bombcell results and exports it as a TSV file that can be viewed in Phy's cluster view tab.

The classification reason shows why each unit was classified as GOOD, NOISE, MUA, or NON-SOMA.

## Usage:
1. Set the configuration parameters below (RUN_MODE, TARGET_PROBE, etc.)
2. Run all cells
3. The notebook will create a `cluster_bc_classificationReason.tsv` file in your Kilosort directory
4. Open the data in Phy - you'll see a new "bc_classificationReason" column in the cluster view

### Imports

In [13]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add parent directory to path for grant_config import
sys.path.insert(0, str(Path.cwd().parent))
from grant_config import load_grant_config
from post_analysis_setup import load_post_analysis_context

import bombcell as bc

### Configuration

In [14]:
# Configuration - CHANGE THESE VALUES FOR YOUR DATA
CONFIG_FILE = r'C:\Users\user\Documents\github\bombcell\py_bombcell\grant\configs\grant_recording_config_reach15_20260201_session007.json'
staging_root = r'H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536'
TARGET_PROBE = 'A' 

# RUN_MODE = 'batch'  # 'batch', 'single_probe', or 'np20_rerun'
# TARGET_PROBE = 'A'  # Only used for single_probe mode

# Optional: Set to True to see detailed information about each unit's classification
VERBOSE = True

#### Setup and Load Data

In [15]:
# Load configuration
cfg = load_grant_config(CONFIG_FILE)

# # Determine staging root based on run mode
# if RUN_MODE == 'batch':
#     staging_root = cfg['default_ks_staging_root']
# elif RUN_MODE == 'np20_rerun':
#     staging_root = cfg['np20_ks_staging_root']
# else:  # single_probe
#     staging_root = cfg['bombcell_singleprobe_root']


# staging_root, save_subdir = mode_to_roots[RUN_MODE]
print('staging_root:', staging_root)
ks_dir = Path(staging_root) / f'kilosort4_{TARGET_PROBE}'
save_path = ks_dir / 'bombcell' 

# # Build paths
# ks_dir = Path(staging_root) / f'kilosort4_{TARGET_PROBE}'
# save_path = ks_dir / 'bombcell'

print('Kilosort directory:', ks_dir)
print('Bombcell save path:', save_path)
print()
print('TSV file will be saved to:', ks_dir / 'cluster_bc_classificationReason.tsv')


# Load the config 
ctx = load_post_analysis_context(CONFIG_FILE)

probe_letters = list(ctx['probeLetters'])
config_probe_letters = probe_letters

print('staging_root:', staging_root)
print('probes:', probe_letters)


staging_root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536
Kilosort directory: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A
Bombcell save path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\bombcell

TSV file will be saved to: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\cluster_bc_classificationReason.tsv
staging_root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536
probes: ['A', 'B', 'C', 'D', 'E', 'F']


In [16]:

manually_set_probes = None # -> ["A","B","C"]

# --------------------------------------------------------------------------
# This is a workaround if for some reason a probe is giving trouble and you want to skip it , or the probe doesnt exist

if manually_set_probes is not None:
    PROBES = manually_set_probes
    print(f'Using manual probe letters: {PROBES}\n')
    print('To use all probes from config, set manually_set_probes = None')
else:
    PROBES = config_probe_letters
    print(f'Using probe letters from config: {probe_letters}')
# --------------------------------------------------------------------------


Using probe letters from config: ['A', 'B', 'C', 'D', 'E', 'F']


In [17]:

# ---- REQUIRED: these must already exist from earlier cells ----
# CONFIG_FILE, RUN_MODE, run_date, run_minute
# load_grant_config, bc

cfg = load_grant_config(CONFIG_FILE)

# if RUN_MODE == "batch":
#     staging_root = cfg["default_ks_staging_root"]
# elif RUN_MODE == "np20_rerun":
#     staging_root = cfg["np20_ks_staging_root"]
# else:  # single_probe
#     staging_root = cfg["bombcell_singleprobe_root"]

# staging_root = Path(str(staging_root) + f"_{run_date}_{run_minute}")

results = []

for probe in PROBES:
    print(f"\n===== Processing Probe {probe} =====")
    ks_dir = Path(staging_root) / f"kilosort4_{probe}"
    save_path = ks_dir / "bombcell"

    print(f"\n===== PROBE {probe} =====")
    print("ks_dir:", ks_dir)
    print("bombcell:", save_path)

    try:
        # 1) Load Bombcell results
        param, quality_metrics, _ = bc.load_bc_results(str(save_path))

        # 2) Unit classifications
        unit_type, unit_type_string = bc.qm.get_quality_unit_type(param, quality_metrics)

        # 3) Build qm_df
        qm_df = pd.DataFrame(quality_metrics).copy()
        qm_df["bombcell_label"] = unit_type_string
        qm_df["unit_index"] = np.arange(len(qm_df))

        # 4) Derive cluster_id (this is the missing piece in your batch cell)
        if "cluster_id" not in qm_df.columns:
            if isinstance(param, dict) and "unique_templates" in param:
                qm_df["cluster_id"] = np.array(param["unique_templates"]).astype(int)
            elif isinstance(quality_metrics, dict) and "phy_clusterID" in quality_metrics:
                qm_df["cluster_id"] = np.array(quality_metrics["phy_clusterID"]).astype(int)
            else:
                qm_df["cluster_id"] = qm_df["unit_index"].astype(int)

        # 5) Minimal export columns for Phy (add more columns here if you want)
        export_df = pd.DataFrame({
            "cluster_id": qm_df["cluster_id"].astype(int),
            "bc_unitType": qm_df["bombcell_label"].astype(str),
        })

        if not export_df["cluster_id"].is_unique:
            raise ValueError("Duplicate cluster_id; Phy merge will be ambiguous")

        out_path = ks_dir / "cluster_bc_unitType.tsv"
        export_df.to_csv(out_path, sep="\t", index=False)
        print("Wrote:", out_path)

        results.append({"probe": probe, "status": "OK", "n_clusters": len(export_df)})

    except Exception as e:
        print("FAILED:", repr(e))
        results.append({"probe": probe, "status": "FAILED", "error": repr(e)})

display(pd.DataFrame(results))



===== Processing Probe A =====

===== PROBE A =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A
bombcell: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\bombcell
Wrote: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\cluster_bc_unitType.tsv

===== Processing Probe B =====

===== PROBE B =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B
bombcell: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B\bombcell
Wrot

,probe,status,n_clusters
0,A,OK,554
1,B,OK,1316
2,C,OK,355
3,D,OK,820
4,E,OK,864
5,F,OK,691


# Add Brain Region to clusterView

#### Configuration Probes 

In [18]:
# ----------------------------
# ROI / Brain-region configuration
# distance is measured from probe TIP in microns
# ----------------------------
probeA_IP_um = (0, 850)        # Interposed nucleus
probeA_SIM_um = (851, 3250)    # Simplex lobule

ROI_END_UM_BY_PROBE = {
    "A": 3450,
    "B": 950,
    "C": 1800,
    "D": 1400,
    "E": 800,
    "F": 1120,
}

PROBE_TO_REGION = {
    "B": "PG",
    "C": "MoP",
    "D": "VaL",
    "E": "SnR",
    "F": "RN",
}

TIP_POSITION = "min_y"  # min_y is the CORRECT choice for probes. 

#### this cell adds the brain region labels using the above configuration data. It also adds a column with the distance of each unit from the probe tip, which is used for classification.q

In [19]:
# New code
from pathlib import Path
import numpy as np
import pandas as pd



def load_or_init_classification_tsv(ks_dir: Path) -> pd.DataFrame:
    target = ks_dir / "cluster_bc_classificationReason.tsv"

    # If already exists, load it
    if target.exists():
        df = pd.read_csv(target, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{target} exists but missing 'cluster_id'")
        return df

    # Otherwise, build it from the best available source
    cluster_info = ks_dir / "cluster_info.tsv"
    cluster_group = ks_dir / "cluster_group.tsv"

    if cluster_info.exists():
        df = pd.read_csv(cluster_info, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{cluster_info} missing 'cluster_id'")
    elif cluster_group.exists():
        df = pd.read_csv(cluster_group, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{cluster_group} missing 'cluster_id'")
    else:
        # last resort: infer cluster ids from spike_clusters.npy
        sc = np.load(ks_dir / "spike_clusters.npy").astype(np.int64)
        df = pd.DataFrame({"cluster_id": np.unique(sc)})

    # Ensure minimal required column set
    if "cluster_id" not in df.columns:
        raise ValueError("Could not construct a df with 'cluster_id'")

    # Write initial file so downstream steps have a consistent target
    df.to_csv(target, sep="\t", index=False)
    return df


cfg = load_grant_config(CONFIG_FILE)

results = []
errors = []


for probe_letter in PROBES:
    ks_dir = Path(staging_root) / f"kilosort4_{probe_letter}"
    results_dir = Path(staging_root) / "roi_brain_region_results.csv"
    tsv_path = ks_dir / "cluster_bc_classificationReason.tsv"

    print(f"\n===== PROBE {probe_letter} =====")
    print("ks_dir:", ks_dir)

    try:

        # if not tsv_path.exists():
        #     raise FileNotFoundError(f"Missing {tsv_path}. Run the earlier export first.")
        # df = pd.read_csv(tsv_path, sep="\t")

        # New code
        df = load_or_init_classification_tsv(ks_dir)
        cluster_ids = df["cluster_id"].astype(int).to_numpy()
        tsv_path = ks_dir / "cluster_bc_classificationReason.tsv"

        if "cluster_id" not in df.columns:
            raise ValueError(f"{tsv_path} missing 'cluster_id' column")

        cluster_ids = df["cluster_id"].astype(int).to_numpy()

        roi_end_um = ROI_END_UM_BY_PROBE.get(probe_letter, None)
        if roi_end_um is None:
            raise ValueError(f"Probe {probe_letter} missing from ROI_END_UM_BY_PROBE")

        # --- Compute primary channel Y per cluster (same logic as your single-probe cell) ---
        spike_clusters = np.load(ks_dir / "spike_clusters.npy").astype(np.int64)
        spike_templates = np.load(ks_dir / "spike_templates.npy").astype(np.int64)
        templates = np.load(ks_dir / "templates.npy")  # (n_templates, n_time, n_channels)
        channel_map = np.load(ks_dir / "channel_map.npy").astype(np.int64).squeeze()
        channel_positions = np.load(ks_dir / "channel_positions.npy")  # (n_channels_total, 2)

        ptp = templates.max(axis=1) - templates.min(axis=1)  # (n_templates, n_channels)

        primary_y = np.full(cluster_ids.shape, np.nan, dtype=float)

        for i, cid in enumerate(cluster_ids):
            idx = np.where(spike_clusters == cid)[0]
            if idx.size == 0:
                continue
            temps = spike_templates[idx]
            t = np.bincount(temps).argmax()
            c_local = int(np.argmax(ptp[t, :]))
            c_phys = int(channel_map[c_local])
            # New code
            c_phys = int(channel_map[c_local])

            # handle 1-based or off-by-one channel_map
            npos = channel_positions.shape[0]
            if c_phys >= npos:
                if (c_phys - 1) >= 0 and (c_phys - 1) < npos:
                    c_phys = c_phys - 1
                else:
                    raise IndexError(
                        f"channel_map index {c_phys} out of bounds for channel_positions size {npos} "
                        f"(probe {probe_letter}, cluster {cid}, template {t}, c_local {c_local})"
                    )

            primary_y[i] = float(channel_positions[c_phys, 1])
            
        if np.all(np.isnan(primary_y)):
            raise RuntimeError("All primary_y are NaN; check KS files")

        # --- ROI by tip distance ---
        tip_y = np.nanmin(primary_y) if TIP_POSITION == "min_y" else np.nanmax(primary_y)
        dist_um = (primary_y - tip_y) if TIP_POSITION == "min_y" else (tip_y - primary_y)

        roi_label = np.where(dist_um <= float(roi_end_um), "IN_ROI", "OUT_ROI")
        in_roi = roi_label == "IN_ROI"

        # --- Brain region mapping ---
        brain_region = np.full(cluster_ids.shape, np.nan, dtype=object)

        if probe_letter == "A":
            in_ip = (dist_um >= probeA_IP_um[0]) & (dist_um <= probeA_IP_um[1]) & in_roi
            in_sim = (dist_um >= probeA_SIM_um[0]) & (dist_um <= probeA_SIM_um[1]) & in_roi
            brain_region[in_ip] = "IP"
            brain_region[in_sim] = "SIM"
            # 
        else:
            region = PROBE_TO_REGION.get(probe_letter, None)
            if region is None:
                raise ValueError(f"Probe {probe_letter} missing from PROBE_TO_REGION")
            brain_region[in_roi] = region

        # --- Write back (add/overwrite columns) ---
        df["bc_ROI"] = roi_label.astype(str)
        df["Brain_Region"] = brain_region

        df.to_csv(tsv_path, sep="\t", index=False)
        print("Updated:", tsv_path)

        if probe_letter == "A":
            regions_to_summarize = ["SIM", "IP"]
        else:
            regions_to_summarize = [PROBE_TO_REGION[probe_letter]]

        for region in regions_to_summarize:
            in_region = brain_region == region
            out_region = ~in_region

            in_ids = cluster_ids[in_region].astype(int).tolist()
            out_ids = cluster_ids[out_region].astype(int).tolist()

            results.append({
                "probe": probe_letter,
                "brain_region": region,
                "n_clusters": int(len(cluster_ids)),
                "IN_ROI_count": int(np.sum(in_region)),
                "OUT_ROI_count": int(np.sum(out_region)),
                "IN_ROI_unitID": in_ids,
                "OUT_ROI_unitID": out_ids,
            })

    except Exception as e:
        print("FAILED:", repr(e))
        errors.append({"probe": probe_letter, "error": repr(e)})

display(pd.DataFrame(results))
results_df = pd.DataFrame(results)
results_df.to_csv(results_dir, index=False)

if errors:
    display(pd.DataFrame(errors))


===== PROBE A =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A
Updated: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\cluster_bc_classificationReason.tsv

===== PROBE B =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B
Updated: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B\cluster_bc_classificationReason.tsv

===== PROBE C =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_C
Updated: H:\Grant\

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,A,SIM,554,385,169,"[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,A,IP,554,169,385,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2..."
2,B,PG,1316,511,805,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[497, 508, 513, 514, 515, 516, 517, 518, 519, ..."
3,C,MoP,355,355,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
4,D,VaL,820,820,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
5,E,SnR,864,226,638,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[221, 222, 227, 229, 230, 231, 232, 233, 234, ..."
6,F,RN,691,202,489,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[199, 201, 202, 205, 206, 207, 208, 209, 210, ..."


### Load in the results df from adding brain regions

In [20]:
results_dir = Path(staging_root) / "roi_brain_region_results.csv"
results_df.to_csv(results_dir, index=False)
results_df

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,A,SIM,554,385,169,"[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,A,IP,554,169,385,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2..."
2,B,PG,1316,511,805,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[497, 508, 513, 514, 515, 516, 517, 518, 519, ..."
3,C,MoP,355,355,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
4,D,VaL,820,820,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
5,E,SnR,864,226,638,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[221, 222, 227, 229, 230, 231, 232, 233, 234, ..."
6,F,RN,691,202,489,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[199, 201, 202, 205, 206, 207, 208, 209, 210, ..."


### Create a dictionary mapping (probe, brain_region) to IN_ROI_unitIDs

In [21]:
results_dir = Path(staging_root) / "roi_brain_region_results.csv"
results_df.to_csv(results_dir, index=False)
results_probe_letters = results_df["probe"].to_list()
results_brain_region = results_df["brain_region"].to_list()
probe_region_pairs = list(zip(results_probe_letters, results_brain_region))
region_inROI_unitIDs = {pair: [] for pair in probe_region_pairs}
for pair in probe_region_pairs:
    probe, brain_region = pair
    region_df = results_df[(results_df["probe"] == probe) & (results_df["brain_region"] == brain_region)]
    if not region_df.empty:
        in_roi_ids = region_df["IN_ROI_unitID"].values[0]
        region_inROI_unitIDs[pair] = in_roi_ids
region_inROI_path =  Path(staging_root)  / "region_inROI_unitIDs.csv"
# save out the region_inROI_unitIDs 

with open(region_inROI_path, "w") as f:
    f.write("probe,brain_region,in_ROI_unitIDs\n")
    for pair, in_roi_ids in region_inROI_unitIDs.items():
        probe, brain_region = pair
        in_roi_ids_str = ";".join(map(str, in_roi_ids))
        f.write(f"{probe},{brain_region},{in_roi_ids_str}\n")

print('probe_letters:', results_probe_letters)
print('brain_regions:', results_brain_region)
print('probe_region_pairs:', probe_region_pairs)
print('region_inROI_unitIDs:', region_inROI_unitIDs)
results_df

probe_letters: ['A', 'A', 'B', 'C', 'D', 'E', 'F']
brain_regions: ['SIM', 'IP', 'PG', 'MoP', 'VaL', 'SnR', 'RN']
probe_region_pairs: [('A', 'SIM'), ('A', 'IP'), ('B', 'PG'), ('C', 'MoP'), ('D', 'VaL'), ('E', 'SnR'), ('F', 'RN')]
region_inROI_unitIDs: {('A', 'SIM'): [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,A,SIM,554,385,169,"[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,A,IP,554,169,385,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 2..."
2,B,PG,1316,511,805,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[497, 508, 513, 514, 515, 516, 517, 518, 519, ..."
3,C,MoP,355,355,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
4,D,VaL,820,820,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
5,E,SnR,864,226,638,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[221, 222, 227, 229, 230, 231, 232, 233, 234, ..."
6,F,RN,691,202,489,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[199, 201, 202, 205, 206, 207, 208, 209, 210, ..."


### veiw the unit IDs that are in the ROI for each probe/brain region pair


In [22]:
# veiw the unit IDs that are in the ROI for each probe/brain region pair
for pair, in_roi_ids in region_inROI_unitIDs.items():
    probe, brain_region = pair
    print(f"Probe {probe}, Brain Region {brain_region}, IN_ROI_unitIDs: {in_roi_ids}")

Probe A, Brain Region SIM, IN_ROI_unitIDs: [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 302, 303, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 3